In [11]:
import logging
from google.cloud import discoveryengine_v1 as discoveryengine
from google.api_core.client_options import ClientOptions

logging.basicConfig(level=logging.INFO)

def get_or_create_identity_mapping_store(
    project_id: str,
    location: str,
    identity_mapping_store_id: str,
) -> discoveryengine.IdentityMappingStore:
  """Get or create an IdentityMappingStore."""
  client_options = ClientOptions(quota_project_id=project_id)
  client_ims = discoveryengine.IdentityMappingStoreServiceClient(client_options=client_options)
  parent_ims = client_ims.location_path(project=project_id, location=location)
  name = f"projects/{project_id}/locations/{location}/identityMappingStores/{identity_mapping_store_id}"

  try:
    request = discoveryengine.GetIdentityMappingStoreRequest(name=name)
    ims = client_ims.get_identity_mapping_store(request=request)
    print(f"Found existing IdentityMappingStore: {ims.name}")
    return ims
  except Exception:
    identity_mapping_store = discoveryengine.IdentityMappingStore()
    request = discoveryengine.CreateIdentityMappingStoreRequest(
        parent=parent_ims,
        identity_mapping_store=identity_mapping_store,
        identity_mapping_store_id=identity_mapping_store_id,
    )
    ims = client_ims.create_identity_mapping_store(request=request)
    print(f"Created IdentityMappingStore: {ims.name}")
    return ims

In [12]:
def create_acl_enabled_data_store(
    project_id: str,
    location: str,
    data_store_id: str,
    display_name: str,
    identity_mapping_store_name: str
) -> discoveryengine.DataStore:
    """Create a new Generic DataStore with ACL enabled and linked to an IMS."""
    from google.api_core.client_options import ClientOptions
    client_options = ClientOptions(quota_project_id=project_id)
    client = discoveryengine.DataStoreServiceClient(client_options=client_options)
    parent = client.collection_path(
        project=project_id,
        location=location,
        collection="default_collection"
    )
    
    data_store = discoveryengine.DataStore(
        display_name=display_name,
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        acl_enabled=True,
        identity_mapping_store=identity_mapping_store_name
    )
    
    request = discoveryengine.CreateDataStoreRequest(
        parent=parent,
        data_store=data_store,
        data_store_id=data_store_id
    )
    
    operation = client.create_data_store(request=request)
    print(f"Waiting for create data store operation {operation.operation.name} to complete...")
    return operation.result()

def get_or_create_acl_enabled_data_store(
    project_id: str,
    location: str,
    data_store_id: str,
    display_name: str,
    identity_mapping_store_name: str
) -> discoveryengine.DataStore:
    from google.api_core.client_options import ClientOptions
    client_options = ClientOptions(quota_project_id=project_id)
    client = discoveryengine.DataStoreServiceClient(client_options=client_options)
    name = client.data_store_path(
        project=project_id,
        location=location,
        data_store=data_store_id
    )
    
    try:
        ds = client.get_data_store(name=name)
        print(f"Found existing DataStore: {ds.name}")
        if not ds.acl_enabled:
            print("WARNING: The existing DataStore does NOT have ACL enabled!")
            print("To use ACLs, you must create a new DataStore or delete the existing one and recreate it.")
        elif not ds.identity_mapping_store:
             print("WARNING: The existing DataStore does NOT have an Identity Mapping Store configured!")
        return ds
    except Exception:
        return create_acl_enabled_data_store(
            project_id=project_id,
            location=location,
            data_store_id=data_store_id,
            display_name=display_name,
            identity_mapping_store_name=identity_mapping_store_name
        )

In [ ]:
PROJECT_ID = "rrolando-agentspace-3245" # Your Gemini Enteprise Project
LOCATION = "global"
IMS_ID = "reddit-identity-store"

# 1. Get or create the shared Identity Mapping Store
ims = get_or_create_identity_mapping_store(PROJECT_ID, LOCATION, IMS_ID)

# 2. Datastore A: Discussions
ds_discussions = get_or_create_acl_enabled_data_store(
    project_id=PROJECT_ID,
    location=LOCATION,
    data_store_id="reddit-discussions-ds",
    display_name="Reddit Enterprise Discussions",
    identity_mapping_store_name=ims.name
)

# 3. Datastore B: Documentation & Repo Files
ds_docs = get_or_create_acl_enabled_data_store(
    project_id=PROJECT_ID,
    location=LOCATION,
    data_store_id="reddit-docs-ds",
    display_name="Reddit Documentation & Files",
    identity_mapping_store_name=ims.name
)

print("Both datastores created and linked successfully!")


Created IdentityMappingStore: projects/889396938283/locations/global/identityMappingStores/reddit-identity-store
Waiting for create data store operation projects/889396938283/locations/global/collections/default_collection/operations/create-data-store-13500258439400407503 to complete...
Waiting for create data store operation projects/889396938283/locations/global/collections/default_collection/operations/create-data-store-5800209730977837885 to complete...
Both datastores created and linked successfully!
